# 01 · The anatomy of a technical chart

A guided tour of how sabia's **trend** and **momentum** features look on a price chart. Each factory returns a `BoundFeature`; `sabia.compute(frame, *features, schema=...)` materializes one strictly-trailing, point-in-time column per feature.

The data here is a **neutral random walk** (`examples/_data.make_ohlcv`) — there is no edge to discover. The goal is simply to read what each indicator measures: trend distance, overbought/oversold, and crossover momentum. Later notebooks simulate data with a *specific* stylized fact (volatility clustering, fat tails, momentum) so a feature can reveal it.

> Requires `pip install plotly nbformat jupyterlab`. Run from this folder.

In [1]:
import sys, pathlib
HERE = pathlib.Path.cwd()
sys.path.insert(0, str(HERE))          # _simulate, _stats live next to this notebook
sys.path.insert(0, str(HERE.parent))   # _data lives one level up, with the example scripts

import numpy as np
import polars as pl
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

pio.renderers.default = "plotly_mimetype+notebook_connected"
pio.templates.default = "plotly_white"

import sabia
from _data import default_schema, make_ohlcv

SCHEMA = default_schema()
print("sabia", sabia.__version__)

sabia 0.2.0


## Bind & compute

Most of these are EWM-recursive and carry a long warm-up (`min_history`), so we compute on a long history and then plot a warmed-up tail. We also reconstruct the SMA/EMA **levels** from the `sma_dist` / `ema_dist` ratios, so the price overlays are literally the trend features.

In [2]:
# A neutral random walk: no edge to find — the point is to read what each indicator measures.
# Features carry a long warm-up (EWM recurrence), so compute on a long history, then plot a tail
# where everything is non-null.
frame = make_ohlcv(n=750)

feats = sabia.compute(
    frame,
    sabia.trend.sma_dist(window=50),
    sabia.trend.ema_dist(span=20),
    sabia.momentum.rsi(period=14),
    sabia.trend.macd(fast=12, slow=26, signal=9),
    sabia.trend.macd_signal(fast=12, slow=26, signal=9),
    sabia.trend.macd_hist(fast=12, slow=26, signal=9),
    sabia.momentum.stoch_k(window=14),
    sabia.momentum.stoch_d(window=14, smooth=3),
    sabia.momentum.cci(window=20),
    schema=SCHEMA,
)

# sma_dist = close / SMA - 1, so SMA = close / (1 + sma_dist); same for the EMA. Reconstructing the
# overlay levels straight from the features keeps the chart honest about what the trend features are.
df = (
    frame.select("timestamp", "open", "high", "low", "close")
    .hstack(feats)
    .with_columns(
        (pl.col("close") / (1.0 + pl.col("sma_dist_50"))).alias("sma_50"),
        (pl.col("close") / (1.0 + pl.col("ema_dist_20"))).alias("ema_20"),
    )
    .tail(260)  # a readable, fully warmed-up window
)
df.select("timestamp", "close", "rsi_14", "macd_12_26_9", "stoch_k_14", "cci_20").tail(3)

timestamp,close,rsi_14,macd_12_26_9,stoch_k_14,cci_20
"datetime[μs, UTC]",f64,f64,f64,f64,f64
2023-01-18 00:00:00 UTC,109.212511,24.66567,-2.970693,4.850843,-128.951914
2023-01-19 00:00:00 UTC,110.260776,29.182877,-3.084679,12.666277,-102.165139
2023-01-20 00:00:00 UTC,109.793122,28.365713,-3.176137,11.300967,-91.04447


## The chart

Five synced panels sharing one time axis: price + moving-average overlays on top, then the oscillators with their conventional thresholds shaded.

In [3]:
t = df.get_column("timestamp").to_list()


def col(name):
    return df.get_column(name).to_list()


fig = make_subplots(
    rows=5,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.03,
    row_heights=[0.40, 0.15, 0.15, 0.15, 0.15],
    subplot_titles=(
        "Price with 50-bar SMA & 20-bar EMA (reconstructed from sma_dist / ema_dist)",
        "RSI(14) — momentum oscillator, 70/30 overbought/oversold",
        "MACD(12,26,9) — line, signal & histogram",
        "Stochastic %K/%D (14,3) — 80/20 bands",
        "CCI(20) — commodity channel index, ±100 bands",
    ),
)

# --- row 1: candlesticks + trend overlays ---
fig.add_trace(
    go.Candlestick(
        x=t, open=col("open"), high=col("high"), low=col("low"), close=col("close"),
        name="price", showlegend=False,
    ),
    row=1, col=1,
)
fig.add_trace(go.Scatter(x=t, y=col("sma_50"), name="SMA 50", line=dict(color="#1f77b4", width=1.3)), row=1, col=1)
fig.add_trace(go.Scatter(x=t, y=col("ema_20"), name="EMA 20", line=dict(color="#ff7f0e", width=1.3)), row=1, col=1)

# --- row 2: RSI ---
fig.add_trace(go.Scatter(x=t, y=col("rsi_14"), name="RSI 14", line=dict(color="#6a3d9a", width=1.2), showlegend=False), row=2, col=1)
fig.add_hrect(y0=70, y1=100, row=2, col=1, fillcolor="#ef5350", opacity=0.10, line_width=0)
fig.add_hrect(y0=0, y1=30, row=2, col=1, fillcolor="#26a69a", opacity=0.10, line_width=0)

# --- row 3: MACD ---
hist = col("macd_hist_12_26_9")
bar_colors = ["#26a69a" if (v or 0.0) >= 0.0 else "#ef5350" for v in hist]
fig.add_trace(go.Bar(x=t, y=hist, name="hist", marker_color=bar_colors, showlegend=False), row=3, col=1)
fig.add_trace(go.Scatter(x=t, y=col("macd_12_26_9"), name="MACD", line=dict(color="#1f77b4", width=1.2), showlegend=False), row=3, col=1)
fig.add_trace(go.Scatter(x=t, y=col("macd_signal_12_26_9"), name="signal", line=dict(color="#ff7f0e", width=1.2), showlegend=False), row=3, col=1)

# --- row 4: stochastic ---
fig.add_trace(go.Scatter(x=t, y=col("stoch_k_14"), name="%K", line=dict(color="#1f77b4", width=1.1), showlegend=False), row=4, col=1)
fig.add_trace(go.Scatter(x=t, y=col("stoch_d_14_3"), name="%D", line=dict(color="#d62728", width=1.1), showlegend=False), row=4, col=1)
fig.add_hrect(y0=80, y1=100, row=4, col=1, fillcolor="#ef5350", opacity=0.10, line_width=0)
fig.add_hrect(y0=0, y1=20, row=4, col=1, fillcolor="#26a69a", opacity=0.10, line_width=0)

# --- row 5: CCI ---
fig.add_trace(go.Scatter(x=t, y=col("cci_20"), name="CCI 20", line=dict(color="#2ca02c", width=1.1), showlegend=False), row=5, col=1)
fig.add_hline(y=100, row=5, col=1, line=dict(color="#ef5350", width=0.8, dash="dot"))
fig.add_hline(y=-100, row=5, col=1, line=dict(color="#26a69a", width=0.8, dash="dot"))

fig.update_layout(
    height=900,
    title="sabia · the anatomy of a technical chart (synthetic random walk)",
    xaxis5_rangeslider_visible=False,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    margin=dict(l=60, r=30, t=90, b=40),
)
for r in range(1, 6):
    fig.update_xaxes(rangeslider_visible=False, row=r, col=1)
fig.show()

## How to read it

- **Price panel** — `sma_dist` / `ema_dist` measure how far price sits above/below a moving average; reconstructed here as overlay levels. The EMA hugs price more tightly than the SMA.
- **RSI(14)** — a bounded `[0,100]` momentum oscillator. Shaded zones mark the canonical >70 (overbought) / <30 (oversold) extremes.
- **MACD(12,26,9)** — the MACD line minus its signal is the histogram; sign flips are the classic crossover signal. `macd`, `macd_signal`, `macd_hist` are three separate features.
- **Stochastic %K/%D** — position of close within its recent high–low range; %D is the smoothed %K. 80/20 bands shaded.
- **CCI(20)** — unbounded; ±100 are the conventional reference lines.

On a random walk these flicker without predictive content — exactly the null case a feature library should make easy to see before you trust a signal on real data.